In [1]:
print("ok")

ok


In [3]:
import os

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [14]:
def load_pdf(data):
    loader = PyPDFDirectoryLoader(data)
    documents = loader.load()
    return documents

In [15]:
extracted_data = load_pdf("../data")

In [ ]:
# extracted_data

In [16]:
def split_text(data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
    text_chunks = text_splitter.split_documents(data)
    return text_chunks

In [17]:
text_chunks = split_text(extracted_data)
print(len(text_chunks))

4536


In [9]:
text_chunks[100].page_content

'of abscesses under the skin. Abscesses near the large\nbowel, particularly around the anus, may be caused by\nany of the numerous bacteria found within the large\nbowel. Brain abscesses and liver abscesses can be caused\nby any organism that can travel there through the circula-\ntion. Bacteria, amoeba, and certain fungi can travel in\nthis fashion. Abscesses in other parts of the body are\ncaused by organisms that normally inhabit nearby struc-\ntures or that infect them. Some common causes of specif-\nic abscesses are:\n• skin abscesses by normal skin flora\n• dental and throat abscesses by mouth flora\n• lung abscesses by normal airway flora, pneumonia\ngerms, or tuberculosis\n• abdominal and anal abscesses by normal bowel flora\nSpecific types of abscesses\nListed below are some of the more common and\nimportant abscesses.\n• Carbuncles and other boils. Skin oil glands (sebaceous\nglands) on the back or the back of the neck are the ones\nusually infected. The most common germ invo

In [4]:
def download_embedding_model():
    model_name = "BAAI/bge-base-en-v1.5"
    model_kwargs = {"device": "cpu"}

    embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)

    return embeddings

In [5]:
embeddings = download_embedding_model()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [21]:
embeddings

HuggingFaceEmbeddings(model_name='BAAI/bge-base-en-v1.5', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [22]:
query_result = embeddings.embed_query("Hello world")
print(len(query_result))

768


In [12]:
query_result[:3]

[-0.03447720408439636, 0.031023239716887474, 0.00673496862873435]

In [18]:
KEY = os.getenv("PINECONE_API_KEY")
index_name = os.getenv("PINECONE_INDEX_NAME")

In [19]:
docsearch = PineconeVectorStore.from_texts(
    texts=[t.page_content for t in text_chunks],
    embedding=embeddings,
    index_name=index_name,
)

In [7]:
docsearch = PineconeVectorStore(index_name=index_name, embedding=embeddings)

In [31]:
query = "What is paracetamol?"
docs = docsearch.similarity_search("paracetamol", k=5)

In [32]:
for doc in docs:
    print("---")
    print(doc.page_content)


---
federal and state laws. A variety of dosage forms are
available, including oral solids, liquids, intravenous and
intrathecal injections, and transcutaneous patches.
NSAIDs, non-steroidal anti-inflammatory drugs, are
effective analgesics even at doses too low to have any
anti-inflammatory effects. There are a number of chemi-
cal classes, but all have similar therapeutic effects and
side effects. Most are appropriate only for oral adminis-
tration; however ketorolac (Toradol) is appropriate for
injection and may be used in moderate to severe pain for
short periods.
Acetaminophen is a non-narcotic analgesic with no
anti-inflammatory properties. It is appropriate for mild to
moderate pain. Although the drug is well tolerated in nor-
mal doses, it may have significant toxicity at high doses.
Because acetaminophen is largely free of side effects at
therapeutic doses, it has been considered the first choice
for mild pain, including that of osteoarthritis.
Recommended dosage
---
acetamino

In [20]:
prompt_template = """
You are a helpful medical assistant.

Use the provided medical context to answer the user's question.

The context may contain:
- synonyms
- abbreviations
- alternate medical names
- related concepts

Provide a complete and natural answer, not just a short phrase.

If two terms refer to the same medicine or condition,
clearly explain that relationship.

Keep answers concise but informative.

Only say you could not find the answer if the context is completely unrelated.

Context:
{context}

Question:
{question}

Answer:
"""

In [21]:
prompt = ChatPromptTemplate.from_template(prompt_template)

In [22]:
llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0.3,
    num_predict=150,
    streaming=True,
)

In [23]:
retriever = docsearch.as_retriever(search_kwargs={"k": 3})

In [24]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [25]:
for chunk in chain.stream("What is paracetamol?"):
    print(chunk, end="", flush=True)

Paracetamol and acetaminophen are actually the same medication, with "paracetamol" being the more commonly used term in some countries, particularly in Europe and Australia. The terms are often used interchangeably to refer to this specific pain-relieving medication.

As mentioned in the provided context, acetaminophen is a medicine used to relieve pain and reduce fever, and it is available without a prescription. It is sold under various brand names, including Tylenol, Panadol, Aspirin Free Anacin, and Bayer Select Maximum Strength Headache Pain Relief Formula, among others.

It's worth noting that while paracetamol and acetaminophen refer to the same medication, there may be slight variations in formulation